# **Wikidata DeepAgent**

## Deep Agent for Wikidata-Based Linked Data Creation

This notebook implements a **Deep Agent–style Wikidata mapping pipeline** that takes domain terms with textual definitions and links them to appropriate **Wikidata items (Q-IDs)** while also assigning a **SKOS mapping type** (exact, close, related).

### 🔧 Main Components

- **Dependency & Environment Setup**
  - Installs and configures `deepagents`, `langchain-openai`, `tavily-python`, and related libraries.
  - Loads API keys (e.g., OpenAI) and sets global configuration for tracing and LLM calls.

- **Wikidata Property & Tooling Layer**
  - Loads a precomputed JSON file of **Wikidata properties** (`wikidata_properties.json`) into a dictionary of `P-id → label`.
  - Defines tools for interacting with Wikidata via its API, including:
    - Searching for candidate entities (Q-IDs) by label and description.
    - Fetching detailed entity information (labels, descriptions, key properties) for disambiguation.
    - Extracting referenced item IDs from claims and mapping property IDs to human-readable labels.

- **SKOS Matching Tool**
  - Reads a curated training file (`autoreconcilitation_training_terms_20251203_formatted.xlsx`) with example mappings.
  - Builds text examples for **exactMatch**, **closeMatch**, and **relatedMatch** using term/definition pairs.
  - Wraps an LLM in a structured-output schema that returns:
    - `exact_match: bool`
    - `close_match: bool`
    - `related_match: bool`
    - `explanation: str`
  - Converts the boolean combination into a single SKOS mapping type (`exact`, `close`, `related`, or `none`) plus a natural-language explanation.

- **System Instructions & Agent Definition**
  - Defines a detailed **system prompt** instructing the agent to:
    - Search for and select the best-fitting Wikidata Q-ID for each term + definition.
    - Use the Wikidata tools to inspect candidate items and construct a concise definition from Wikidata facts.
    - Compare this constructed definition with the provided one and decide on a SKOS mapping type.
    - Keep an internal log of tried identifiers to avoid repetition.
  - Declares a `Wikimapping` Pydantic model with fields:
    - `qid` – the chosen Wikidata identifier (e.g., `Q159`)
    - `skos` – the SKOS mapping label (`exact`, `close`, `related`, `none`)
    - `explanation` – reasoning behind the mapping decision.
  - Creates the **Deep Agent** (`agent_wiki`) via `create_deep_agent`, connecting:
    - the LLM model,
    - the Wikidata tools, and
    - the SKOS matching tool,
    - with `Wikimapping` as the structured response format.

- **Interactive Example**
  - Demonstrates the agent on a single test term (e.g., `"Berlin"` with a short definition) to:
    - select a Q-ID,
    - classify the SKOS relation,
    - and show the reasoning for the mapping.

- **Batch Processing for a Real Dataset**
  - Defines `batch_run_wikidata_match(input_xlsx, output_xlsx)` to run the agent over a full **KIDA** term set stored in an Excel workbook.
  - Reads:
    - a `Mappings` sheet (terms and existing mapping columns),
    - a `Used definitions` sheet (term → definition).
  - For each term marked as not mapped (e.g., `"No wiki match"`):
    - retrieves its definition,
    - invokes the Wikidata agent,
    - parses the structured result (`qid`, `skos`, `explanation`),
    - writes back:
      - `Mapped Term` (Q-ID),
      - `Mapped ID` (full Wikidata URL),
      - `Mapped Type` / `Additional ID` (e.g., `"wikidata"`),
      - `SKOS_matching` and `SKOS_explanation`.
  - Saves the updated `Mappings` and `Used definitions` back to a new Excel file for further validation and analysis.
  - Provides two run classes:
    - first class: general mapping run (x2),
    - second class: a modified prompt focusing more on **related** mappings to capture non-exact linkages (x2).

- **Result Inspection**
  - Includes a small utility to **interactively select and display** the results of a chosen run (or a combined file) as a scrollable HTML table inside the notebook.

### 📌 Purpose

Overall, this code implements a **Wikidata-aware agent** that automates **linked data creation** by:
1. Resolving free-text terms + definitions to **Wikidata Q-IDs**, and  
2. Classifying the semantic relationship using **SKOS mapping types**,  
producing a structured, explainable mapping output suitable for knowledge graph integration and ontology alignment workflows.


# **Agent implementation with all steps exlained**

## **Intsallation and importing of dependencies**

*   Deep Agent package from Langchain
*   Langchain community tool
*   Communication tools with  OpenAI

Optional

*   Communication package with Google in case Gemini models are preferred

API keys are imported from the Colab stored userdata




In [1]:
%reset -f
!pip install deepagents tavily-python langchain-openai langchain_community
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.2/388.2 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests=

In [32]:
#@title Necessary imports

from google.colab import userdata # For API keys
import os # for environmental variables
from io import StringIO # to act like a file but you don’t want to create an actual file on disk.

## Structure processings
import json
from typing import Optional, Iterable, List, Dict, Any, Set, Tuple
import datetime
import requests
import re
import pandas as pd

# LLM-related imports
from langchain.chat_models import init_chat_model
#from langchain_google_genai import ChatGoogleGenerativeAI
from deepagents import create_deep_agent

from typing import Optional, Literal, Tuple
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent

from langchain_core.messages import SystemMessage, HumanMessage

# Part needed to inspect xlsx
from IPython.display import display, HTML


In [4]:
# @title Imports using colab stored keys
#from google.colab import userdata
OPENAI_API_KEY=userdata.get('api-key-MicRisk')
Langsmith_key=userdata.get('langsmith')
# ANTHROPIC_API_KEY=userdata.get('anthropic')
# GEMINI_API_KEY=userdata.get('gemini')

In [5]:
# @title Specifying global variables  for Langsmith tracing and LLM calls
#import os
# os.environ["GEMINI_API_KEY"] =GEMINI_API_KEY
os.environ["OPENAI_API_KEY"] =OPENAI_API_KEY
os.environ["LANGSMITH_API_KEY"] =Langsmith_key
os.environ["LANGSMITH_TRACING"] ="true"
os.environ["LANGSMITH_PROJECT"] ="KIDA_data"
os.environ["LANGSMITH_ENDPOINT"]="https://eu.api.smith.langchain.com"
# os.environ["ANTHROPIC_API_KEY"] =ANTHROPIC_API_KEY
#os.environ["TAVILY_API_KEY"] =TAVILY_API_KEY

## **Specifying Agent tools**

In [19]:
# @title Uploading the stored list of wikidata properties. See Wikidata_tools\Property_list_retrieval

# Path to your uploaded Wikidata properties file
FILE_PATH = "/content/wikidata_properties.json"

# Your existing minimal PROPERTY_LABELS (overrides)
PROPERTY_LABELS = {}

def load_property_labels_from_file(path: str) -> dict:
    """
    Load property labels from JSON.

    Expected format:
      {
         "P10": "video",
         "P101": "field of work",
         ...
      }

    Returns:
        dict: { "Pxxx": "label", ... }
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # The uploaded file is already a dict {pid: label}
    if isinstance(data, dict):
        return data

    raise ValueError("Unexpected JSON structure: expected a dictionary of PID→label.")


# Load the big Wikidata property dictionary
DYNAMIC_PROPERTY_LABELS = load_property_labels_from_file(FILE_PATH)

# Merge → existing manual PROPERTY_LABELS take precedence
for pid, label in DYNAMIC_PROPERTY_LABELS.items():
    PROPERTY_LABELS.setdefault(pid, label)

print("Loaded", len(DYNAMIC_PROPERTY_LABELS), "properties from JSON.")
print("Total PROPERTY_LABELS now:", len(PROPERTY_LABELS))

# Quick preview
example_keys = list(PROPERTY_LABELS.keys())[:20]
print({k: PROPERTY_LABELS[k] for k in example_keys})

Loaded 13065 properties from JSON.
Total PROPERTY_LABELS now: 13065
{'P10': 'video', 'P101': 'field of work', 'P102': 'member of political party', 'P103': 'native language', 'P105': 'taxon rank', 'P106': 'occupation', 'P108': 'employer', 'P109': 'signature', 'P110': 'illustrator', 'P111': 'measured physical quantity', 'P112': 'founded by', 'P113': 'airline hub', 'P114': 'airline alliance', 'P115': 'home venue', 'P117': 'chemical structure', 'P118': 'league or competition', 'P119': 'place of burial', 'P121': 'item operated', 'P122': 'basic form of government', 'P123': 'publisher'}


In [20]:
# @title Defining tools for the communication with wikidata.See Deep_Agent\Deep_Agent_implementation\Wikidata_tools
# from typing import Iterable, List, Dict, Any
# import datetime
# import requests

# Wikidata API base URL
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

HEADERS = {
    "User-Agent": "MyReActAgent/0.1 Linked_data"
}


def _extract_time_string(wikidata_time: str) -> str:
    """
    Convert Wikidata time format '+1955-07-26T00:00:00Z' to '1955-07-26'.
    Falls back to the original string if parsing fails.
    """
    try:
        if wikidata_time.startswith("+") or wikidata_time.startswith("-"):
            wikidata_time = wikidata_time[1:]
        dt = datetime.datetime.fromisoformat(wikidata_time.replace("Z", "+00:00"))
        return dt.date().isoformat()
    except Exception:
        return wikidata_time


def _collect_referenced_item_ids(entities: Dict[str, Any]) -> List[str]:
    """
    From an entities dict (wbgetentities result), collect referenced item IDs (Qxxx)
    from the subset of properties we care about.
    """
    referenced_ids = set()
    for entity_id, entity in entities.items():
        claims = entity.get("claims", {})
        for pid in PROPERTY_LABELS.keys():
            for claim in claims.get(pid, []):
                mainsnak = claim.get("mainsnak", {})
                datavalue = mainsnak.get("datavalue", {})
                value = datavalue.get("value")
                if (
                    isinstance(value, dict)
                    and value.get("entity-type") == "item"
                    and "id" in value
                ):
                    referenced_ids.add(value["id"])
    # remove the original entity ids; we only need labels for referenced ones
    referenced_ids -= set(entities.keys())
    return list(referenced_ids)


def _get_entities(ids: Iterable[str], language: str = "en") -> Dict[str, Any]:
    """
    Helper to call wbgetentities for a list of Q-ids and return the 'entities' map.
    """
    ids_list = list(ids)
    if not ids_list:
        return {}

    params = {
        "action": "wbgetentities",
        "ids": "|".join(ids_list),
        "format": "json",
        "languages": language,
        "props": "labels|descriptions|claims",
    }
    response = requests.get(
        WIKIDATA_API_URL,
        params=params,
        headers=HEADERS,  # <-- important for avoiding 403
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()
    return data.get("entities", {})


def _get_entity_labels(ids: Iterable[str], language: str = "en") -> Dict[str, str]:
    """
    Get labels for item IDs (e.g. Q5, Q30) in the given language.
    """
    ids_list = list(ids)
    if not ids_list:
        return {}

    params = {
        "action": "wbgetentities",
        "ids": "|".join(ids_list),
        "format": "json",
        "languages": language,
        "props": "labels",
    }
    response = requests.get(
        WIKIDATA_API_URL,
        params=params,
        headers=HEADERS,  # <-- important for avoiding 403
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()
    entities = data.get("entities", {})

    labels = {}
    for eid, entity in entities.items():
        label_obj = entity.get("labels", {}).get(language)
        if label_obj:
            labels[eid] = label_obj.get("value")
    return labels

def get_wikidata_definition(
    entity_id: str,
    language: str = "en",
) -> Optional[Dict[str, Any]]:
    """
    Tool: Given a single Wikidata entity ID (e.g. 'Q42'),
    construct an enriched definition for that term using Wikidata entity data.

    Returns:
        A dict like:
        {
          "id": "Q42",
          "label": "...",
          "description": "...",
          "definition": "...",
          "url": "https://www.wikidata.org/wiki/Q42",
          "facts": { ... }
        }
        or None if the entity cannot be retrieved.
    """
    if not entity_id:
        return None

    # 1) Get entity data for this single ID
    entities = _get_entities([entity_id], language=language)
    if entity_id not in entities:
        # Nothing found for this ID
        return None

    entity = entities[entity_id]

    # 2) Collect referenced item ids for properties of interest, to get nicer labels
    referenced_item_ids = _collect_referenced_item_ids({entity_id: entity})
    referenced_labels = _get_entity_labels(referenced_item_ids, language=language)

    def _value_to_string(datavalue: Dict[str, Any]) -> str:
        """
        Convert a Wikidata datavalue into a human-readable string, using
        referenced_labels when possible.
        """
        if not datavalue:
            return ""

        value = datavalue.get("value")
        vtype = datavalue.get("type")

        if vtype == "wikibase-entityid" and isinstance(value, dict):
            eid = value.get("id")
            return referenced_labels.get(eid, eid or "")

        if vtype == "time" and isinstance(value, dict):
            return _extract_time_string(value.get("time", ""))

        if vtype == "globecoordinate" and isinstance(value, dict):
            lat = value.get("latitude")
            lon = value.get("longitude")
            if lat is not None and lon is not None:
                return f"{lat}, {lon}"
            return str(value)

        # Fallback: just stringify
        return str(value)

    labels = entity.get("labels", {})
    descriptions = entity.get("descriptions", {})
    claims = entity.get("claims", {})

    label = labels.get(language, {}).get("value") or entity_id
    description = descriptions.get(language, {}).get("value")

    # Build structured "facts" from selected properties
    facts: Dict[str, List[str]] = {}

    for pid, human_label in PROPERTY_LABELS.items():
        prop_claims = claims.get(pid, [])
        values: List[str] = []
        for claim in prop_claims:
            mainsnak = claim.get("mainsnak", {})
            datavalue = mainsnak.get("datavalue")
            if datavalue:
                s = _value_to_string(datavalue)
                if s:
                    values.append(s)
        if values:
            facts[human_label] = values

    # Construct definition text
    definition_parts: List[str] = []
    if description:
        definition_parts.append(description.rstrip("."))

    if facts:
        fact_strings = []
        for human_label, values in facts.items():
            joined_vals = ", ".join(values)
            fact_strings.append(f"{human_label}: {joined_vals}")
        definition_parts.append("Key facts: " + "; ".join(fact_strings))

    if not definition_parts:
        definition_parts.append(
            f"{label} (no textual description available in Wikidata)."
        )

    definition = ". ".join(definition_parts).strip()
    if not definition.endswith("."):
        definition += "."

    return {
        "id": entity_id,
        "label": label,
        "description": description,
        "definition": definition,
        "url": f"https://www.wikidata.org/wiki/{entity_id}",
        "facts": facts,
    }

def resolve_qids_and_pids_in_definition(
    enriched_result: Dict[str, Any],
    language: str = "en",
) -> Dict[str, Any]:
    """
    Given a single enriched entity from `get_wikidata_definition`,
    replace:
      - Q-IDs (e.g. 'Q183') with their Wikidata labels
      - P-IDs (e.g. 'P2076') with their property labels from PROPERTY_LABELS

    Replacement is applied to:
      - the 'definition' string
      - all string values inside the 'facts' dict

    Args:
        enriched_result: output of get_wikidata_definition (a dict for one entity)
        language: label language to use when resolving Q-IDs

    Returns:
        A NEW dict with Q- and P-IDs replaced by labels.
        (Original dict is not mutated.)
    """
    if not enriched_result:
        return enriched_result

    qid_pattern = re.compile(r"\bQ\d+\b")
    pid_pattern = re.compile(r"\bP\d+\b")

    # 1) Collect all Q-IDs that appear in definition or facts
    all_qids: Set[str] = set()

    # From definition text
    definition = enriched_result.get("definition") or ""
    all_qids.update(qid_pattern.findall(definition))

    # From facts (only string values)
    facts = enriched_result.get("facts") or {}
    if isinstance(facts, dict):
        for vals in facts.values():
            for v in vals:
                if isinstance(v, str):
                    all_qids.update(qid_pattern.findall(v))

    # 2) Resolve Q-IDs -> labels using Wikidata (in chunks, though usually few)
    entity_labels: Dict[str, str] = {}
    if all_qids:
        qid_list = list(all_qids)
        chunk_size = 50

        for i in range(0, len(qid_list), chunk_size):
            chunk = qid_list[i : i + chunk_size]
            chunk_labels = _get_entity_labels(chunk, language=language)
            entity_labels.update(chunk_labels)

    # 3) Helper to replace Q-IDs and P-IDs in any string
    def _replace_ids_in_text(text: str) -> str:
        # First replace Q-IDs with entity labels
        text = qid_pattern.sub(
            lambda m: entity_labels.get(m.group(0), m.group(0)),
            text,
        )
        # Then replace P-IDs with property labels from PROPERTY_LABELS
        text = pid_pattern.sub(
            lambda m: PROPERTY_LABELS.get(m.group(0), m.group(0)),
            text,
        )
        return text

    # 4) Build a new dict with replaced IDs
    new_item = dict(enriched_result)  # shallow copy

    # Replace in definition
    definition = enriched_result.get("definition")
    if isinstance(definition, str):
        new_item["definition"] = _replace_ids_in_text(definition)

    # Replace inside facts strings
    facts = enriched_result.get("facts")
    if isinstance(facts, dict):
        new_facts: Dict[str, list] = {}
        for key, vals in facts.items():
            new_vals: list = []
            for v in vals:
                if isinstance(v, str):
                    new_vals.append(_replace_ids_in_text(v))
                else:
                    new_vals.append(v)
            new_facts[key] = new_vals
        new_item["facts"] = new_facts

    return new_item

def WikidataEntityDetails (q: str):
     """
     Fetch full Wikidata details for a given entity (e.g. 'Q159').
     Input should be Q-ID only, and output is the JSON of that entity
     """

     raw = get_wikidata_definition(q) # Get definition containing Q-ids and P-ids
     resolved = resolve_qids_and_pids_in_definition(raw) # Enrich the definition
     return resolved


def get_nested_value(o: dict, path: list) -> any:
    """
    Safely walk through nested dicts and lists by keys/indexes.
    Returns None if any KeyError, IndexError, or TypeError occurs.
    """
    current = o
    for key in path:
        try:
            current = current[key]
        except (KeyError, IndexError, TypeError):
            return None
    return current


def WikidataEntitySearch(
    search: str,
    entity_type: str = "item",
    url: str = "https://www.wikidata.org/w/api.php",
    user_agent_header: str = 'DeepWikidataMapper/0.1',
    srqiprofile: str = None,
) -> Optional[str]:
    """
    Search Wikidata entities for a given query.

    Args:
        search: Text to search for (e.g. "Berlin").
        entity_type: Type of entity to search ('item' or 'property').
        url: Wikidata API URL.
        user_agent_header: User-Agent string for requests.
        srqiprofile: Search profile for Wikidata API.

    Returns:
        The Q-ID or P-ID of the top result, or an error message if not found.
    """
    headers = {"Accept": "application/json"}
    if user_agent_header is not None:
        headers["User-Agent"] = user_agent_header

    if entity_type == "item":
        srnamespace = 0
        srqiprofile = "classic_noboostlinks" if srqiprofile is None else srqiprofile
    elif entity_type == "property":
        srnamespace = 120
        srqiprofile = "classic" if srqiprofile is None else srqiprofile
    else:
        raise ValueError("entity_type must be either 'property' or 'item'")

    params = {
        "action": "query",
        "list": "search",
        "srsearch": search,
        "srnamespace": srnamespace,
        "srlimit": 1,
        "srqiprofile": srqiprofile,
        "srwhat": "text",
        "format": "json",
    }

    response = requests.get(url, headers=headers, params=params)

    if response.status_code == 200:
        title = get_nested_value(response.json(), ["query", "search", 0, "title"])
        if title is None:
            return f"I couldn't find any {entity_type} for '{search}'. Please rephrase your request and try again"
        # if there is a prefix, strip it off
        return title.split(":")[-1]
    else:
        return "Sorry, I got an error. Please try again."


# def WikidataEntityDetails (q: str):
#     """
#     Fetch full Wikidata details for a given entity (e.g. 'Russia Q159').
#     Input should be the label plus Q-ID, and output is the JSON of that entity
#     """
#     wikidata = WikidataQueryRun(api_wrapper=WikidataAPIWrapper())

#     return wikidata.run(q)




### Defining SKOS matching tool

In [21]:
# @title Defining tools for the communication with wikidata.See Deep_Agent\Deep_Agent_implementation\SKOS_matching_tool

df=pd.read_excel("/content/autoreconcilitation_training_terms_20251203_formatted.xlsx")

def build_match_pairs(
    df: pd.DataFrame,
    match_name: str,
    label_col: str,
    desc_col: str,
    term_col: str = "term",
    def_col: str = "definition",
):
    """
    Build plain-text pairs for a given SKOS match type.
    Returns a full text block as a string instead of printing.
    """
    buffer = StringIO()
    buffer.write(f"========== {match_name} pairs ==========\n\n")

    subset = df.dropna(subset=[label_col])

    for n, row in enumerate(subset.itertuples(index=False), start=1):
        term_a = getattr(row, term_col)
        def_a  = getattr(row, def_col)
        term_b = getattr(row, label_col)
        def_b  = getattr(row, desc_col)

        buffer.write(f"{n}) Term A: {term_a}\n")
        buffer.write(f"   Definition A: {def_a}\n")
        buffer.write(f"   Term B: {term_b}\n")
        buffer.write(f"   Definition B: {def_b}\n\n")

    return buffer.getvalue()

exact_text = build_match_pairs(
    df,
    match_name="exactMatch",
    label_col="exactMatch_label",
    desc_col="exactMatch_description"
)

close_text = build_match_pairs(
    df,
    match_name="closeMatch",
    label_col="closeMatch_label",
    desc_col="closeMatch_description"
)

related_text = build_match_pairs(
    df,
    match_name="relatedMatch",
    label_col="relatedMatch_label",
    desc_col="relatedMatch_description"
)

class SKOSMatch(BaseModel):
    """SKOS-style semantic relationship between two concepts."""

    exact_match: bool = Field(
        default=None,
        description=(
            f"""True if the two concepts can be used interchangeably across schemes.
They denote the same real-world concept, even if the wording differs.
This is symmetric and transitive.

EXACT MATCH EXAMPLES:
{exact_text}
"""
        ),
    )

    close_match: bool = Field(
        default=None,
        description=(
            f"""True if the two concepts are very similar and usually substitutable in most contexts,
but not strictly equivalent. Not transitive.

CLOSE MATCH EXAMPLES:
{close_text}
"""
        ),
    )

    related_match: bool = Field(
        default=None,
        description=(
            f"""True if the two concepts are associated but not synonymous.
Represents a non-hierarchical 'see also' relation.

RELATED MATCH EXAMPLES:
{related_text}
"""
        ),
    )

    explanation: Optional[str] = Field(
        default=None,
        description=(
            """Short explanation of why the chosen semantic relationship holds,
based on comparing terms and definitions."""
        ),
    )


# agent = create_agent(
#     model="gpt-5.1",
#     response_format=SKOSMatch,
# )

llm_skos = ChatOpenAI(
    model="gpt-5.1",
    temperature=0,
)
structured_llm_skos = llm_skos.with_structured_output(SKOSMatch)


def classify_skos_match(term_a: str, gen_def: str, term_b: str, onto_def: str):
    """
    Function to classify semantic relationships between two concepts into the
    SKOS concept: exact, close and related. The output is the matching type and
    the explanation
    """
    prompt=f"""
        You are comparing semantic similarities between two concepts. Each concept
        is represented by a term and its definition. Provide a concise summary
        explaining the semantic relationship between the concepts.

        Concept A:\n
          Term: {term_a}
          Definition (generated): {gen_def}
        Concept B:
          Term: {term_b}
          Definition (ontology): {onto_def}

      """

    messages = [
        HumanMessage(content=prompt),
    ]

    data: SKOSMatch = structured_llm_skos.invoke(messages)


     # Decide mapping_type with priority: exact > close > related
    if data.exact_match:
        mapping_type = "exact"
    elif data.close_match:
        mapping_type = "close"
    elif data.related_match:
        mapping_type = "related"
    else:
        mapping_type = "none"

    return {
        "mapping_type": mapping_type,
        "explanation": data.explanation or ""
    }


## **Specifying LLM for the agent**

In [12]:
# from langchain.chat_models import init_chat_model
# #from langchain_google_genai import ChatGoogleGenerativeAI
# from deepagents import create_deep_agent

#model = init_chat_model(model="claude-opus-4-5-20251101")
model = init_chat_model(model="gpt-5.1")
#model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## **Specifying system instructions**

In [22]:
research_instructions = f"""You task is to match the terms with valid identifiers from wikidata.

First find the identifier that may fit, then use the tools to get additional information about this identifier and based on this information construct the consice definition
of the term linked to this identifier. The wikidata label does not need to match the searhched term exactly, but definitions of the term and wikidata labels should be in one of these broad categories

Exact matching: The two concepts can be used interchangeably across schemes.They denote the same real-world concept, even if the wording differs.
{exact_text}

Close matching: The two concepts are very similar and usually substitutable in most contexts, but not strictly equivalent.
{close_text}

Related matching: The two concepts are associated but not synonymous. Represents a non-hierarchical 'see also' relation.
{related_text}


Then compare the constructed definition and the provided. If these definitions match, then return the found identifier. If not, continue the search among other identifiers.

Keep track on what identifiers you tried to avoid repetitive tries"""



In [23]:
research_instructions

"You task is to match the terms with valid identifiers from wikidata.\n\nFirst find the identifier that may fit, then use the tools to get additional information about this identifier and based on this information construct the consice definition\nof the term linked to this identifier. The wikidata label does not need to match the searhched term exactly, but definitions of the term and wikidata labels should be in one of these broad categories \n\nExact matching: The two concepts can be used interchangeably across schemes.They denote the same real-world concept, even if the wording differs.\n========== exactMatch pairs ==========\n\n1) Term A: Campylobacter jejuni\n   Definition A: Gram-negative bacterial species that can cause diarrhoea in people\n   Term B: Campylobacter jejuni\n   Definition B: species of bacterium\n\n2) Term A: Cow's milk\n   Definition A: Whole, fresh, lacteal secretion obtained by the complete milking of one or more healthy cows\n   Term B: cow's milk\n   Definit

## **Construct the wikidata agent**

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class Wikimapping(BaseModel):
    """Wkidata mapping output"""
    qid: str = Field(description="The identificator number, Q-id. Example: Q159")
    skos: str = Field(description="SKOS_matching. SKOS matching between original term definition and the definition/description of the identified label from wikidata. Example: exact")
    explanation: str = Field(description="SKOS_matching_logic. The explanation for SKOS matching logic retrieved from explanation field of SKOS matching tool" )

agent_wiki = create_deep_agent(model=model,
    tools=[WikidataEntitySearch, WikidataEntityDetails, classify_skos_match],
    system_prompt=research_instructions,
    response_format=Wikimapping
)

# **Example test for a single example entry**

In [25]:
term_test= 'Berlin'
definition_test = ' City in the US'

question =f"""What is the best fitting Q-identifier the term {term_test} which definition {definition_test} matches with the label from the wikidata .
As a reply only provide
(i) identificator number,
(ii) SKOS matching between original term definition and the definition/description of the identified label from wikidata
(iii) Explanation for SKOS matching.

SKOS matching and corresponding explanation should be identified with the corresponding tool -

If no proper match is found, you may adjust the search query and try with other identifiers.

If no proper identifier is found after 10 iterations, return "No wiki match".In that case SKOS matching is not needed

Store your logic in a separete file using tools for file read, write and edit. Export this file at the end of the run in txt format """

# Phrase: Store your logic in a separete file using tools for file read, write and edit. Export this file at the end of the run in txt format can be removed to reduce token usage.
# Now is done for the transparency purposes

result = agent_wiki.invoke({"messages": [{"role": "user", "content": question}]})

# Print the agent's response
print(result["messages"][-1].content)

{"qid":"Q1569850","skos":"exact","explanation":"Both concepts denote the same city: Berlin located in the United States. The original definition (“City in the US”) is general but fully consistent with the Wikidata description (“city in Green Lake and Waushara counties, Wisconsin, United States”); it just omits the more specific county-level details, so they can be used interchangeably in context."}


### **Details of the example agent run**


*   See the all details of the agent run in [the shared LangSmith trace](https://eu.smith.langchain.com/public/605cfc76-f44f-4674-b440-a9cd61e2d749/r)

*   Below is the copy of the logic the agent stored
```text
     1	Search log for term: Berlin (definition: City in the US)
     2
     3	Tried Q1005623 (New Berlin) - description: city in Waukesha County, Wisconsin, United States
     4	Reasoning: Label is 'New Berlin', not 'Berlin', and the city name differs, so this is only a related concept, not an exact or close match to generic 'Berlin, city in the US'. Will search for a better match that directly has label 'Berlin' in the US.
     5
     6	Tried Q1569850 (Berlin, Wisconsin) - description: city in Green Lake and Waushara counties, Wisconsin, United States
     7	Reasoning: Label is exactly 'Berlin' and it is explicitly a city in the United States, matching the generic definition 'City in the US'. This is taken as an exactMatch
```

*   Here is the copy of the final agent output
```text
{"qid":"Q1569850","skos":"exact","explanation":"Both concepts denote the same city: Berlin located in the United States. The original definition (“City in the US”) is general but fully consistent with the Wikidata description (“city in Green Lake and Waushara counties, Wisconsin, United States”); it just omits the more specific county-level details, so they can be used interchangeably in context."}
```





# **Test for the real data : KIDA dataset**

In [39]:
# @title This run was executed twice to allow for the agent to map terms not mapped during the first run and labeled with "No wiki match"
def batch_run_wikidata_match(input_xlsx: str, output_xlsx: str):

    ##################### updated to process unverified #############################
    # Load both sheets
    mappings_df = pd.read_excel(input_xlsx, sheet_name="Mappings")
    defs_df     = pd.read_excel(input_xlsx, sheet_name="Used definitions")

    # Build a dict: term → definition
    def_map = defs_df.set_index("Term")["Definition"].to_dict()

    # Make sure columns exist
    if "Mapped Term" not in mappings_df.columns:
        mappings_df["Mapped Term"] = pd.NA
    if "Mapped Type" not in mappings_df.columns:
        mappings_df["Mapped Type"] = pd.NA
    if "Mapped ID" not in mappings_df.columns:
        mappings_df["Mapped ID"] = pd.NA
    if "Additional ID" not in mappings_df.columns:
        mappings_df["Additional ID"] = pd.NA
    if "SKOS_matching" not in mappings_df.columns:
        mappings_df["SKOS_matching"] = pd.NA
    if "SKOS_matching_logic" not in mappings_df.columns:
        mappings_df["SKOS_explanation"] = pd.NA

    # mask_not_mapped = mappings_df["Mapped Term"].isna()

    # # Convert Mapped Type to string so .str works
    # mapped_type_str = mappings_df["Mapped Type"].astype("string")
    # mask_unverified = mapped_type_str.str.contains("Unverified", case=False, na=False) # used in the multiagent system, not needed for a single agent

    # mask_to_process = mask_not_mapped | mask_unverified
    #to_process = mappings_df[mask_to_process].copy()

    mapped_term_str = mappings_df["Mapped Term"].astype("string")
    mask_to_process = mapped_term_str.str.contains("No wiki match", case=False, na=False) # for the second run

    to_process = mappings_df[mask_to_process].copy()

    # For each term with no mapped term yet or unverified...
    for idx, row in to_process.iterrows():
        term = row["Term"]
        definition = def_map.get(term, "")
        if not isinstance(definition, str):
            definition = str(definition)
        definition = definition.strip()

        if not definition:
            # Skip if no definition available
            continue

        # Clear memory for a fresh agent run (assuming you have a memory object)
        if "memory" in globals():
            memory.clear()


#         question =f"""What is the best fitting Q-identifier from wikidata for the term {term} which definition is :" {definition} ".
# As a reply only provide the identificator number. If no proper match is found, return "No wiki match"""

#         question =f"""What is the best fitting Q-identifier the term {term} which definition {definition} matches with the label from the wikidata .
# As a reply only provide the identificator number. If no proper match is found, you may adjust the search query and try with other identifiers.
# If no proper identifier is found after 10 iterations, return "No wiki match". """

        QUESTION_QUERY =f"""What is the best fitting Q-identifier the term {term} which definition {definition} matches with the label from the wikidata .
As a reply only provide the identificator number, SKOS matching between (i) original term and definition and (ii) the label and its definition
from wikidata, and the explanation for SKOS matching. SKOS matching and corresponding explanation should be identified with the corresponding tool -

If no proper match is found, you may adjust the search query and try with other identifiers.

If no proper identifier is found after 10 iterations, return "No wiki match".In that case SKOS matching is not needed

Store your logic in a separete file using tools for file read, write and edit. Export this file at the end of the run in txt format """


        result = agent_wiki.invoke({"messages": [{"role": "user", "content": QUESTION_QUERY}]})


        result_text = result["messages"][-1].content

        # Optionally keep for debugging:
        # mappings_df.at[idx, "LLM Raw Result"] = result_text

        # if result_text.upper().startswith("Q"):
        #     # Q-ID found → construct URL and fill other fields
        #     mappings_df.at[idx, "Mapped Term"] = result_text
        #     url = f"https://www.wikidata.org/wiki/{result_text}"
        #     mappings_df.at[idx, "Mapped ID"]     = url
        #     mappings_df.at[idx, "Mapped Type"]   = "wikidata"
        #     mappings_df.at[idx, "Additional ID"] = "wikidata"
        #     #mappings_df.at[idx, "SKOS_matching"] = ""


        if isinstance(result_text, str):
          parsed = json.loads(result_text)
          qid = parsed .get("qid")
          skos_match = parsed .get("skos", "")
          explanation = parsed .get("explanation", "")
        else:
          qid = None
          skos_match = ""
          explanation = ""

        if qid:
    # Q-ID → Mapped Term
          mappings_df.at[idx, "Mapped Term"] = qid

    # Build Wikidata URL
          url = f"https://www.wikidata.org/wiki/{qid}"
          mappings_df.at[idx, "Mapped ID"]     = url
          mappings_df.at[idx, "Mapped Type"]   = "wikidata"
          mappings_df.at[idx, "Additional ID"] = "wikidata"

    # SKOS-matching word → SKOS_matching (or your column name)
          mappings_df.at[idx, "SKOS_matching"] = skos_match
          mappings_df.at[idx, "SKOS_explanation"] = explanation

    # Save both sheets back to Excel
    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        mappings_df.to_excel(writer, sheet_name="Mappings", index=False)
        defs_df.to_excel(writer, sheet_name="Used definitions", index=False)

    print(f"Finished batch run; results saved to {output_xlsx}")


In [40]:
# @title saving the results
batch_run_wikidata_match(
    input_xlsx="/content/Terms_wiki_test_agent_completed_formatted.xlsx",
    output_xlsx="/content/Terms_wiki_test_agent_completed_formatted_mapped.xlsx")

/tmp/ipython-input-3613022518.py:105: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Q170065' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mappings_df.at[idx, "Mapped Term"] = qid
/tmp/ipython-input-3613022518.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'https://www.wikidata.org/wiki/Q170065' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mappings_df.at[idx, "Mapped ID"]     = url
/tmp/ipython-input-3613022518.py:110: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'wikidata' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mappings_df.at[idx, "Mapped Type"]   = "wikidata"
/tmp/ipython-input-3613022518.py:1

Finished batch run; results saved to /content/Terms_wiki_test_agent_completed_formatted_mapped.xlsx


In [51]:
# @title The run with the modified prompt to shift the focus on related matches, as previously no "close" or exact matches were found. The run was executed twice
# import pandas as pd
# import re
# import json

#os.environ["LANGSMITH_PROJECT"] ="KIDA_dataset" # group traces

def batch_run_wikidata_match(input_xlsx: str, output_xlsx: str):

    ##################### updated to process unverified #############################
    # Load both sheets
    mappings_df = pd.read_excel(input_xlsx, sheet_name="Mappings")
    defs_df     = pd.read_excel(input_xlsx, sheet_name="Used definitions")

    # Build a dict: term → definition
    def_map = defs_df.set_index("Term")["Definition"].to_dict()

    # Make sure columns exist
    if "Mapped Term" not in mappings_df.columns:
        mappings_df["Mapped Term"] = pd.NA
    if "Mapped Type" not in mappings_df.columns:
        mappings_df["Mapped Type"] = pd.NA
    if "Mapped ID" not in mappings_df.columns:
        mappings_df["Mapped ID"] = pd.NA
    if "Additional ID" not in mappings_df.columns:
        mappings_df["Additional ID"] = pd.NA
    if "SKOS_matching" not in mappings_df.columns:
        mappings_df["SKOS_matching"] = pd.NA
    if "SKOS_matching_logic" not in mappings_df.columns:
        mappings_df["SKOS_explanation"] = pd.NA



    # Convert Mapped Type to string so .str works
    mapped_term_str = mappings_df["Mapped Term"].astype("string")
    mask_to_process = mapped_term_str.str.contains("No wiki match", case=False, na=False)

    to_process = mappings_df[mask_to_process].copy()

    # For each term with no mapped term yet or unverified...
    for idx, row in to_process.iterrows():
        term = row["Term"]
        definition = def_map.get(term, "")
        if not isinstance(definition, str):
            definition = str(definition)
        definition = definition.strip()

        if not definition:
            # Skip if no definition available
            continue

        # Clear memory for a fresh agent run (assuming you have a memory object)
        if "memory" in globals():
            memory.clear()


#         question =f"""What is the best fitting Q-identifier from wikidata for the term {term} which definition is :" {definition} ".
# As a reply only provide the identificator number. If no proper match is found, return "No wiki match"""

#         question =f"""What is the best fitting Q-identifier the term {term} which definition {definition} matches with the label from the wikidata .
# As a reply only provide the identificator number. If no proper match is found, you may adjust the search query and try with other identifiers.
# If no proper identifier is found after 10 iterations, return "No wiki match". """

        QUESTION_QUERY =f"""What is the best fitting Q-identifier the term {term} which definition {definition} matches with the label from the wikidata .
As a reply only provide the identificator number, SKOS matching between (i) original term and definition and (ii) the label and its definition
from wikidata, and the explanation for SKOS matching. SKOS matching and corresponding explanation should be identified with the corresponding tool -

Previously no matches were found for the term :{term}, definition: {definition}. Thus, focus more on related matches, meaning that the label may only be associated with the
term and knowledge domain.

If no proper identifier is found after 10 iterations, return "No wiki match".In that case SKOS matching is not needed

Store your logic in a separete file using tools for file read, write and edit. Export this file at the end of the run in txt format """


        result = agent_wiki.invoke({"messages": [{"role": "user", "content": QUESTION_QUERY}]})


        result_text = result["messages"][-1].content

        # Optionally keep for debugging:
        # mappings_df.at[idx, "LLM Raw Result"] = result_text

        # if result_text.upper().startswith("Q"):
        #     # Q-ID found → construct URL and fill other fields
        #     mappings_df.at[idx, "Mapped Term"] = result_text
        #     url = f"https://www.wikidata.org/wiki/{result_text}"
        #     mappings_df.at[idx, "Mapped ID"]     = url
        #     mappings_df.at[idx, "Mapped Type"]   = "wikidata"
        #     mappings_df.at[idx, "Additional ID"] = "wikidata"
        #     #mappings_df.at[idx, "SKOS_matching"] = ""


        if isinstance(result_text, str):
          parsed = json.loads(result_text)
          qid = parsed .get("qid")
          skos_match = parsed .get("skos", "")
          explanation = parsed .get("explanation", "")
        else:
          qid = None
          skos_match = ""
          explanation = ""

        if qid:
    # Q-ID → Mapped Term
          mappings_df.at[idx, "Mapped Term"] = qid

    # Build Wikidata URL
          url = f"https://www.wikidata.org/wiki/{qid}"
          mappings_df.at[idx, "Mapped ID"]     = url
          mappings_df.at[idx, "Mapped Type"]   = "wikidata"
          mappings_df.at[idx, "Additional ID"] = "wikidata"

    # SKOS-matching word → SKOS_matching (or your column name)
          mappings_df.at[idx, "SKOS_matching"] = skos_match
          mappings_df.at[idx, "SKOS_explanation"] = explanation

    # Save both sheets back to Excel
    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        mappings_df.to_excel(writer, sheet_name="Mappings", index=False)
        defs_df.to_excel(writer, sheet_name="Used definitions", index=False)

    print(f"Finished batch run; results saved to {output_xlsx}")

In [52]:
batch_run_wikidata_match(
    input_xlsx="/content/Terms_wiki_test_agent_completed_formatted_mapped_4.xlsx",
    output_xlsx="/content/Terms_wiki_test_agent_completed_formatted_mapped_5.xlsx")

Finished batch run; results saved to /content/Terms_wiki_test_agent_completed_formatted_mapped_5.xlsx


In [34]:
# @title Inspect the results of the runs

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# --- Ask user which run to inspect ---
print("Which run do you want to inspect?")
print("  • Runs 1–2: general agent runs")
print("  • Runs 3–4: focused on related-mapping evaluation")
print("  • Run 5: combined results from all runs\n")

# --- Get run number ---
while True:
    try:
        run_number = int(input("Please enter a run number (1–5): ").strip())
        if 1 <= run_number <= 5:
            break
        else:
            print("❗ Please enter a number between 1 and 5.")
    except ValueError:
        print("❗ Invalid input — please enter an integer between 1 and 5.")

# --- Build filename dynamically ---
if run_number == 5:
    xlsx_path = "/content/Terms_wiki_test_agent_combined.xlsx"
else:
    xlsx_path = f"/content/Terms_wiki_test_agent_run_{run_number}.xlsx"

print(f"\nLoading file: {xlsx_path}")

# --- Check file existence ---
if not os.path.exists(xlsx_path):
    raise FileNotFoundError(
        f"❗ File not found: {xlsx_path}\n"
        f"Make sure the file is uploaded or the path is correct."
    )

# --- Load Excel ---
df = pd.read_excel(xlsx_path)

# --- Create scrollable HTML table ---
scrollable_html = (
    "<div style='max-height:600px; overflow:auto; border:1px solid lightgray;'>"
    + df.to_html(index=True, justify='left')
    + "</div>"
)

# --- Display ---
display(HTML(scrollable_html))



Which run do you want to inspect?
  • Runs 1–2: general agent runs
  • Runs 3–4: focused on related-mapping evaluation
  • Run 5: combined results from all runs

Please enter a run number (1–5): 5

Loading file: /content/Terms_wiki_test_agent_combined.xlsx


,Term,Mapped Term,Mapped ID,Mapped Type,Additional ID,SKOS_matching,SKOS_explanation
0,Aetiologic agent,Q170065,https://www.wikidata.org/wiki/Q170065,wikidata,wikidata,exact,"Using the classify_skos_match logic: Term A: 'Aetiologic agent' with definition 'the microorganisms or pathogens responsible for causing the diseases or conditions being studied (i.e., pathogens identified as causes of non-malarial febrile illnesses in Africa)' denotes any infectious microorganism or agent that causes the disease under investigation. Term B (Wikidata Q170065): 'pathogen' with definition 'biological entity that causes disease in its host, which is typically an infectious microorganism or agent, such as a virus, bacterium, protozoan, prion, viroid, or fungus' denotes any disease-causing infectious biological agent. Both concepts refer to the same real-world notion of disease-causing infectious agents and are substitutable in this biomedical context. Hence the relation is SKOS exactMatch."
1,References,Q10358455,https://www.wikidata.org/wiki/Q10358455,wikidata,wikidata,exact,"Both the user’s concept and the Wikidata item describe the same real-world entity: a bibliographic reference/citation that provides the essential identifying details of a source used in a scholarly work. The user frames it as the entries listed in the References section with full details so readers can locate the original materials; Wikidata defines it as the minimum data needed to identify the literary source of a piece of information. Despite the user mentioning the section context, the core concept is the individual bibliographic reference, so these can be used interchangeably in scholarly communication, justifying an exact match."
2,Bacteria,Q10876,https://www.wikidata.org/wiki/Q10876,wikidata,wikidata,exact,"Both the user’s term and definition describe bacteria as the broad biological group/domain of bacterial microorganisms, including genera like Escherichia and pathogenic species that infect humans and animals. The Wikidata item Q10876 is the taxon 'bacteria', defined as a domain of microorganisms. Despite the user definition adding examples and mentioning pathogenicity, both refer to the same taxonomic/biological concept, so the mapping is an exact match (per classify_skos_match)."
3,Streptococcus spp.,Q190161,https://www.wikidata.org/wiki/Q190161,wikidata,wikidata,exact,"[functions.classify_skos_match] Both the original term ""Streptococcus spp."" (described as bacteria associated with febrile illness in African regions) and the Wikidata item ""Streptococcus"" refer to the same biological genus of bacteria. The ""spp."" notation simply indicates multiple species within that genus, which is fully consistent with a genus-level taxon concept, so they can be treated as the same concept at the intended level of abstraction."
4,Escherichia spp.,Q25419,https://www.wikidata.org/wiki/Q25419,wikidata,wikidata,close,"Mapping type: closeMatch (via classify_skos_match). Term A: “Escherichia spp.” with definition “Escherichia spp. are bacterial pathogens found in blood samples contributing to various infections.” Term B (Wikidata): “Escherichia coli” (Q25419), defined as an enteric, rod-shaped, gram‑negative bacterium, taxon rank species, parent taxon Escherichia. E. coli is a single species within the broader genus Escherichia (Escherichia spp.) and is indeed a common pathogenic Escherichia found in blood and other clinical samples. However, Escherichia spp. properly refers to all species in the genus, not only E. coli. Thus B is a specific member of the broader group denoted by A, making them closely related and often substitutable in clinical shorthand but not strictly equivalent, so the appropriate SKOS relation is closeMatch rather than exactMatch."
5,Typhoidal Salmonella spp.,Q83319,https://www.wikidata.org/wiki/Q83319,wikidata,wikidata,relatedMatch,"Tool functions.classify_skos_match: Concept A is the group of typhoidal Salmonella bacteria (the causative agents), whil